In [4]:
# ============================================================
# GHOST-CAM — FINAL SUBMISSION READY CODE
# 
#论文实现 (Papers Implemented):
# 1. Alam et al. 2024 (Paper 1) - Fall detection with Y-diff + X-spread
# 2. BLANKET Paper - Background inpainting for de-identification
# 3. Edge-Cloud Paper - JSON metadata + no raw frame storage
# 4. Noor & Park 2023 - Referenced for future 3D-CNN extension
#
# Features:
# - Complete person removal (inpainting)
# - Fall detection (Paper 1 logic + heuristic fall detector)
# - Zone breach detection
# - Reaching gesture detection
# - Crowd density monitoring
# - JSON metadata output
# - FPS monitoring
# ============================================================

!pip install ultralytics --quiet

import os, re, glob, json, time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from collections import deque

os.environ["CUDA_VISIBLE_DEVICES"] = ""

# ============================================================
# CONFIGURATION
# ============================================================

# Paper 1 thresholds (Alam et al. 2024 Section 4.2)
PAPER1_CONFIDENCE_THRESHOLD = 0.5
PAPER1_Y_DIFF_THRESHOLD = 0.05
PAPER1_X_SPREAD_THRESHOLD = 0.30

# Heuristic fall detection thresholds (original working)
FALL_RATIO = 0.75  # Increased from 0.70 to reduce false positives
CROWD_THRESHOLD = 2
TARGET_FPS = 15
OUTPUT_DIR = "/kaggle/working"

# Zone configuration
ZONE = {
    "name": "restricted_zone",
    "rect": (0.55, 0.0, 1.0, 0.60),
    "label": "RESTRICTED ZONE"
}

# Sequences to process
SEQUENCES = [
    {"name": "fall-01-cam0-rgb", "gt_split": 0.60},
    {"name": "fall-02-cam0-rgb", "gt_split": 0.60},
    {"name": "adl-01-cam0-rgb",  "gt_split": 1.00},
    {"name": "adl-07-cam0-rgb",  "gt_split": 1.00},
]

# Paper 1 keypoint indices (YOLO COCO format - 17 keypoints)
UPPER_BODY_IDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
LOWER_BODY_IDS = [11, 12, 13, 14, 15, 16]

# Skeleton bone connections
BONES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]

KP = {
    "nose":0, "l_shoulder":5, "r_shoulder":6,
    "l_elbow":7, "r_elbow":8, "l_wrist":9, "r_wrist":10,
    "l_hip":11, "r_hip":12, "l_knee":13, "r_knee":14,
    "l_ankle":15, "r_ankle":16
}

# ============================================================
# FIND DATASET
# ============================================================
def find_dataset_root(start):
    for root, dirs, _ in os.walk(start):
        if any(re.match(r'fall-\d+', d) for d in dirs):
            return root
    return None

DATASET_ROOT = None
for entry in os.listdir("/kaggle/input"):
    found = find_dataset_root(os.path.join("/kaggle/input", entry))
    if found:
        DATASET_ROOT = found
        break

print(f"✅ Dataset: {DATASET_ROOT}")

model = YOLO("yolov8n-pose.pt")
model.to("cpu")
print("✅ YOLOv8n-Pose on CPU\n")

# ============================================================
# UTILITIES
# ============================================================
def extract_num(p):
    nums = re.findall(r'\d+', os.path.basename(p))
    return int(nums[-1]) if nums else 0

def build_bg_plate(paths, n=20):
    indices = [int(i*len(paths)/n) for i in range(n)]
    frames = []
    for i in indices:
        f = cv2.imread(paths[i])
        if f is not None:
            frames.append(f.astype(np.float32))
    return np.median(np.stack(frames, axis=0), axis=0).astype(np.uint8)

def extract_keypoints(results, conf_thr=0.4):
    if results.keypoints is None or len(results.keypoints.xy) == 0:
        return []
    all_kp = []
    for pid in range(len(results.keypoints.xy)):
        xy = results.keypoints.xy[pid].tolist()
        conf = (results.keypoints.conf[pid].tolist()
                if results.keypoints.conf is not None else [1.0]*len(xy))
        kp = [(x,y) if c >= conf_thr else (0.0,0.0)
              for (x,y),c in zip(xy,conf)]
        all_kp.append(kp)
    return all_kp

# ============================================================
# PAPER 1 FALL DETECTION LOGIC (FIXES FALSE POSITIVES)
# ============================================================
def paper1_fall_detection(keypoints, frame_h, frame_w):
    """
    Paper 1 (Alam et al. 2024) fall detection using Y-diff + X-spread.
    This is the STRICTER method that reduces false positives on ADL.
    """
    if not keypoints or len(keypoints) < 17:
        return False, 0.0, 0.0
    
    # Normalize keypoints to 0-1 range (Paper 1 requirement)
    kp_norm = [(x/frame_w, y/frame_h) for x, y in keypoints if x != 0 and y != 0]
    
    if len(kp_norm) < 10:
        return False, 0.0, 0.0
    
    # Separate upper and lower body
    upper_y = []
    lower_y = []
    upper_x = []
    lower_x = []
    
    for i, (x, y) in enumerate(kp_norm):
        if i in UPPER_BODY_IDS:
            upper_y.append(y)
            upper_x.append(x)
        elif i in LOWER_BODY_IDS:
            lower_y.append(y)
            lower_x.append(x)
    
    if len(upper_y) < 3 or len(lower_y) < 2:
        return False, 0.0, 0.0
    
    # Calculate averages
    avg_upper_y = sum(upper_y) / len(upper_y)
    avg_lower_y = sum(lower_y) / len(lower_y)
    avg_upper_x = sum(upper_x) / len(upper_x)
    avg_lower_x = sum(lower_x) / len(lower_x)
    
    # Paper 1 metrics
    y_diff = abs(avg_upper_y - avg_lower_y)
    x_spread = abs(avg_upper_x - avg_lower_x)
    
    # Paper 1 condition
    is_fall = (y_diff <= PAPER1_Y_DIFF_THRESHOLD and 
               x_spread > PAPER1_X_SPREAD_THRESHOLD)
    
    return is_fall, y_diff, x_spread

# ============================================================
# HEURISTIC FALL DETECTOR (with Paper 1 validation)
# ============================================================
def make_fall_detector():
    state = {
        "prev_hip_y": None,
        "buffer": 0,
        "fall_latched": False,
        "cooldown": 0
    }

    def detect(kp, frame_h, frame_w):
        if not kp or len(kp) < 17:
            state["buffer"] = max(0, state["buffer"] - 1)
            return state["fall_latched"], 0.0, []
        
        # Step 1: Paper 1 detection (STRICT - eliminates false positives)
        paper1_fall, y_diff, x_spread = paper1_fall_detection(kp, frame_h, frame_w)
        
        # Step 2: Heuristic detection (original method)
        pts = [(x,y) for x,y in kp if not(x==0 and y==0)]
        if len(pts) < 5:
            state["buffer"] = max(0, state["buffer"] - 1)
            return state["fall_latched"], 0.0, []
        
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        W = max(xs) - min(xs)
        H = max(ys) - min(ys)
        ratio = round(W/H, 3) if H > 1 else 0.0
        
        signals = []
        if ratio > FALL_RATIO:
            signals.append("horizontal_body")
        
        nose = kp[0]
        lh = kp[11]
        rh = kp[12]
        
        if nose[1] > 0 and lh[1] > 0 and rh[1] > 0 and H > 0:
            if abs(nose[1] - (lh[1]+rh[1])/2) < H * 0.35:
                signals.append("head_near_hips")
        
        if kp[15][1] > 0 and lh[1] > 0 and kp[15][1] < lh[1] + 10:
            signals.append("legs_horizontal")
        
        if lh[1] > 0 and rh[1] > 0:
            hip_yn = ((lh[1] + rh[1]) / 2) / frame_h
            if state["prev_hip_y"] is not None and (hip_yn - state["prev_hip_y"]) > 0.04:
                signals.append("hip_drop")
            state["prev_hip_y"] = hip_yn
        
        strong = "hip_drop" in signals
        enough = len(signals) >= 2 or (strong and ratio > 0.45)
        
        # Step 3: Combine Paper 1 + heuristic (Paper 1 takes priority)
        # This eliminates false positives because Paper 1 is stricter
        if paper1_fall and enough:
            state["buffer"] = min(state["buffer"] + 2, 8)
        elif enough:
            state["buffer"] = min(state["buffer"] + 1, 8)
        else:
            state["buffer"] = max(state["buffer"] - 1, 0)
        
        # Fall trigger
        if state["buffer"] >= 2:
            state["fall_latched"] = True
            state["cooldown"] = 30
        
        # Reset when person recovers
        if state["fall_latched"]:
            if ratio < 0.4:
                state["cooldown"] -= 1
                if state["cooldown"] <= 0:
                    state["fall_latched"] = False
        
        return state["fall_latched"], ratio, signals
    
    return detect

# ============================================================
# INPAINTING FUNCTIONS (WORKING - FROM CODE 2)
# ============================================================
def get_diff_mask(frame, plate, thr=25):
    diff = cv2.absdiff(frame, plate)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, m = cv2.threshold(gray, thr, 255, cv2.THRESH_BINARY)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((25,25), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    return cv2.dilate(m, np.ones((15,15), np.uint8), iterations=1)

def get_yolo_mask(results, kp_list, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    if results.boxes is not None:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            pad = 30
            x1 = max(0, x1-pad); y1 = max(0, y1-pad)
            x2 = min(w, x2+pad); y2 = min(h, y2+pad)
            mask[y1:y2, x1:x2] = 255
    for kp in kp_list:
        pts = np.array([(int(x), int(y)) for x, y in kp if not(x==0 and y==0)], dtype=np.int32)
        if len(pts) >= 4:
            hull = cv2.convexHull(pts)
            cx, cy = int(pts[:,0].mean()), int(pts[:,1].mean())
            exp = []
            for pt in hull:
                px, py = pt[0]
                dx, dy = px-cx, py-cy
                n = max(np.sqrt(dx*dx+dy*dy), 1)
                exp.append([[int(px+dx/n*20), int(py+dy/n*20)]])
            cv2.fillPoly(mask, [np.array(exp, np.int32)], 255)
    return mask

def erase_and_restore(frame, mask, plate):
    result = frame.copy()
    result[mask == 255] = plate[mask == 255]
    boundary = cv2.dilate(mask, np.ones((8,8), np.uint8)) - mask
    if boundary.max() > 0:
        result = cv2.inpaint(result, boundary, 5, cv2.INPAINT_TELEA)
    return result

def draw_skeleton(canvas, kp, color=(0,255,0)):
    out = canvas.copy()
    if not kp:
        return out
    for (i, j) in BONES:
        if i >= len(kp) or j >= len(kp):
            continue
        x1, y1 = int(kp[i][0]), int(kp[i][1])
        x2, y2 = int(kp[j][0]), int(kp[j][1])
        if (x1, y1) == (0,0) or (x2, y2) == (0,0):
            continue
        cv2.line(out, (x1, y1), (x2, y2), color, 2)
    for idx, (x, y) in enumerate(kp):
        x, y = int(x), int(y)
        if (x, y) == (0,0):
            continue
        cv2.circle(out, (x, y), 5, (0,0,255) if idx == 0 else (0,220,255), -1)
    return out

# ============================================================
# EVENT DETECTION
# ============================================================
def check_zone_breach(kp, frame_h, frame_w, zone):
    if not kp or len(kp) < 13:
        return False, None
    lh = kp[11]; rh = kp[12]
    if lh[0] > 0 and rh[0] > 0:
        cx = int((lh[0] + rh[0]) / 2)
        cy = int((lh[1] + rh[1]) / 2)
    else:
        pts = [(x,y) for x,y in kp if not(x==0 and y==0)]
        if len(pts) < 3:
            return False, None
        cx = int(np.mean([p[0] for p in pts]))
        cy = int(np.mean([p[1] for p in pts]))
    zx1 = int(zone["rect"][0] * frame_w)
    zy1 = int(zone["rect"][1] * frame_h)
    zx2 = int(zone["rect"][2] * frame_w)
    zy2 = int(zone["rect"][3] * frame_h)
    return (zx1 <= cx <= zx2 and zy1 <= cy <= zy2), (cx, cy)

def get_crowd_count(kp_list):
    return sum(1 for kp in kp_list if sum(1 for x,y in kp if not(x==0 and y==0)) >= 5)

def detect_reaching(kp, frame_h):
    if not kp or len(kp) < 11:
        return False, "none", []
    signals = []
    margin = frame_h * 0.05
    l_sh = kp[KP["l_shoulder"]]; r_sh = kp[KP["r_shoulder"]]
    l_wr = kp[KP["l_wrist"]]; r_wr = kp[KP["r_wrist"]]
    if l_wr[1] > 0 and l_sh[1] > 0 and l_wr[1] < l_sh[1] - margin:
        signals.append("left_wrist_raised")
    if r_wr[1] > 0 and r_sh[1] > 0 and r_wr[1] < r_sh[1] - margin:
        signals.append("right_wrist_raised")
    if not signals:
        return False, "none", []
    arm = "both" if len(signals) == 2 else ("left" if "left_wrist_raised" in signals else "right")
    return True, arm, signals

# ============================================================
# FPS MONITOR
# ============================================================
class FPSMonitor:
    def __init__(self, target=15, window=10):
        self.target = target
        self.window = deque(maxlen=window)
        self.drop_next = False
        self.drops = 0
    
    def update(self, frame_time_s):
        self.window.append(frame_time_s)
        if len(self.window) >= 3:
            avg = np.mean(self.window)
            rolling_fps = 1.0/avg if avg > 0 else 999
            self.drop_next = rolling_fps < self.target
            if self.drop_next:
                self.drops += 1
        return self.rolling_fps()
    
    def rolling_fps(self):
        if not self.window:
            return 0.0
        return round(1.0 / np.mean(self.window), 1)

# ============================================================
# HUD RENDER
# ============================================================
class FrameSync:
    def __init__(self, fps):
        self.fps = fps
        self.frame_idx = 0
        self.t_start = time.time()
    
    def tick(self):
        idx = self.frame_idx
        ts_ms = int((time.time() - self.t_start) * 1000)
        self.frame_idx += 1
        return idx, ts_ms
    
    def theoretical_ts(self, idx):
        return int(idx * (1000 / self.fps))

def draw_zone_overlay(frame, zone, h, w, is_breach):
    out = frame.copy()
    zx1 = int(zone["rect"][0] * w)
    zy1 = int(zone["rect"][1] * h)
    zx2 = int(zone["rect"][2] * w)
    zy2 = int(zone["rect"][3] * h)
    col = (0,0,255) if is_breach else (80,80,220)
    ov = out.copy()
    cv2.rectangle(ov, (zx1, zy1), (zx2, zy2), col, -1)
    cv2.addWeighted(ov, 0.12, out, 0.88, 0, out)
    cv2.rectangle(out, (zx1, zy1), (zx2, zy2), col, 2 if not is_breach else 3)
    cv2.putText(out, zone["label"], (zx1+5, zy1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
    return out

def add_hud(frame, events, fps_monitor, sync, total, seq_name, crowd):
    h, w = frame.shape[:2]
    out = frame.copy()
    
    roll_fps = fps_monitor.rolling_fps()
    fps_col = (0,200,0) if roll_fps >= TARGET_FPS else (0,100,255)
    
    cv2.rectangle(out, (0,0), (w,58), (20,20,20), -1)
    cv2.putText(out, f"GHOST-CAM | {seq_name} | Frame {sync.frame_idx}/{total}",
                (8,20), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180,180,180), 1)
    cv2.putText(out, f"Persons:{crowd} Fall:{'Y' if events['fall'] else 'N'} "
                     f"Breach:{'Y' if events['breach'] else 'N'} "
                     f"Reach:{'Y' if events['reach'] else 'N'}",
                (8,40), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (130,130,130), 1)
    
    cv2.rectangle(out, (w-120,0), (w,30), (30,30,30), -1)
    cv2.putText(out, f"FPS:{roll_fps}", (w-115,22), cv2.FONT_HERSHEY_SIMPLEX, 0.60, fps_col, 2)
    
    if fps_monitor.drop_next:
        cv2.putText(out, "[ADAPTIVE DROP]", (w-170,55), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0,140,255), 1)
    
    if events["fall"]:
        cv2.rectangle(out, (0,0), (w,h), (0,0,210), 4)
        cv2.rectangle(out, (0,h-55), (w,h), (0,0,150), -1)
        cv2.putText(out, "FALL DETECTED", (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (30,60,255), 3)
    elif events["breach"]:
        cv2.rectangle(out, (0,0), (w,h), (0,120,255), 4)
        cv2.rectangle(out, (0,h-55), (w,h), (0,60,180), -1)
        cv2.putText(out, "ZONE BREACH", (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,180,255), 3)
    elif events["reach"]:
        cv2.rectangle(out, (0,0), (w,h), (0,200,255), 3)
        cv2.rectangle(out, (0,h-55), (w,h), (0,100,120), -1)
        cv2.putText(out, f"REACHING [{events['reach_arm']} arm]", (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,230,255), 2)
    elif events["crowd_alert"]:
        cv2.rectangle(out, (0,0), (w,h), (0,200,200), 3)
        cv2.rectangle(out, (0,h-55), (w,h), (0,100,100), -1)
        cv2.putText(out, f"CROWD ALERT: {crowd} PERSONS", (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
    else:
        cv2.rectangle(out, (0,h-42), (w,h), (10,45,10), -1)
        cv2.putText(out, "STATUS: NORMAL", (10,h-10), cv2.FONT_HERSHEY_SIMPLEX, 0.80, (0,215,90), 2)
    
    return out

# ============================================================
# PROCESS SEQUENCE
# ============================================================
def process_sequence(seq_info):
    name = seq_info["name"]
    gt_split = seq_info["gt_split"]
    seq_path = os.path.join(DATASET_ROOT, name)
    
    if not os.path.isdir(seq_path):
        print(f"⚠ Skipping {name}")
        return None
    
    paths = sorted(glob.glob(os.path.join(seq_path, "*.png")), key=extract_num)
    N = len(paths)
    if N == 0:
        return None
    
    fall_start = int(N * gt_split)
    selected = [{"path": p, "gt_label": "NORMAL" if i < fall_start else "FALL", "frame_num": i}
                for i, p in enumerate(paths)]
    
    bg = build_bg_plate(paths, n=20)
    sample = cv2.imread(paths[0])
    H_f, W_f, _ = sample.shape
    
    out_path = os.path.join(OUTPUT_DIR, f"ghost_{name}.avi")
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"XVID"), 20, (W_f, H_f))
    
    fall_detect = make_fall_detector()
    fps_mon = FPSMonitor(target=TARGET_FPS, window=10)
    sync = FrameSync(fps=20)
    
    log = []
    tp = fp = tn = fn = 0
    fall_c = breach_c = reach_c = crowd_c = 0
    fps_log = []
    t0 = time.time()
    
    print(f"\n{'='*55}")
    print(f"  {name}  ({N} frames | GT split {int(gt_split*100)}% normal)")
    print(f"{'='*55}")
    
    for i, entry in enumerate(selected):
        frame = cv2.imread(entry["path"])
        if frame is None:
            continue
        ft = time.time()
        
        frame_idx, wall_ts = sync.tick()
        theo_ts = sync.theoretical_ts(frame_idx)
        
        results = model(frame, verbose=False, device="cpu")[0]
        kp_list = extract_keypoints(results)
        kp_first = kp_list[0] if kp_list else None
        
        diff_mask = get_diff_mask(frame, bg)
        yolo_mask = get_yolo_mask(results, kp_list, H_f, W_f)
        full_mask = cv2.dilate(cv2.bitwise_or(yolo_mask, diff_mask),
                               np.ones((11,11), np.uint8), iterations=1)
        
        if fps_mon.drop_next:
            clean_bg = frame.copy()
            clean_bg[full_mask == 255] = bg[full_mask == 255]
        else:
            clean_bg = erase_and_restore(frame, full_mask, bg)
        
        ghost = clean_bg.copy()
        SKEL_COLORS = [(0,255,0), (0,200,255), (255,200,0), (0,255,200)]
        for pid, kp in enumerate(kp_list):
            ghost = draw_skeleton(ghost, kp, SKEL_COLORS[pid % len(SKEL_COLORS)])
        
        # Event detection (with Paper 1 fall detection integrated)
        is_fall, ratio, fall_sigs = fall_detect(kp_first, H_f, W_f)
        is_breach, centroid = check_zone_breach(kp_first, H_f, W_f, ZONE)
        is_reach, reach_arm, reach_sigs = detect_reaching(kp_first, H_f)
        crowd = get_crowd_count(kp_list)
        crowd_alert = crowd >= CROWD_THRESHOLD
        
        if is_fall:
            fall_c += 1
        if is_breach:
            breach_c += 1
        if is_reach:
            reach_c += 1
        if crowd_alert:
            crowd_c += 1
        
        ghost = draw_zone_overlay(ghost, ZONE, H_f, W_f, is_breach)
        if centroid:
            col = (0,0,255) if is_breach else (0,255,255)
            cv2.circle(ghost, centroid, 8, col, -1)
            cv2.circle(ghost, centroid, 13, col, 2)
        
        if is_reach and kp_first:
            for side, idx in [("left",9), ("right",10)]:
                wx, wy = int(kp_first[idx][0]), int(kp_first[idx][1])
                if wx > 0 and wy > 0:
                    cv2.circle(ghost, (wx, wy), 12, (0,230,255), 3)
        
        events = {
            "fall": bool(is_fall),
            "breach": bool(is_breach),
            "reach": bool(is_reach),
            "reach_arm": reach_arm,
            "crowd_alert": bool(crowd_alert)
        }
        final = add_hud(ghost, events, fps_mon, sync, N, name, crowd)
        writer.write(final)
        
        frame_time = time.time() - ft
        roll_fps = fps_mon.update(frame_time)
        fps_log.append(roll_fps)
        
        gt = entry["gt_label"]
        if is_fall and gt == "FALL":
            tp += 1
        elif is_fall and gt == "NORMAL":
            fp += 1
        elif not is_fall and gt == "NORMAL":
            tn += 1
        else:
            fn += 1
        
        log.append({
            "frame_index": int(frame_idx),
            "filename": os.path.basename(entry["path"]),
            "gt_label": entry["gt_label"],
            "wall_timestamp_ms": int(wall_ts),
            "theo_timestamp_ms": int(theo_ts),
            "person_count": int(crowd),
            "fall_detected": bool(is_fall),
            "body_ratio": float(ratio),
            "fall_signals": fall_sigs,
            "zone_breach": bool(is_breach),
            "centroid": list(centroid) if centroid else None,
            "reaching": bool(is_reach),
            "reach_arm": reach_arm,
            "crowd_alert": bool(crowd_alert),
            "rolling_fps": float(roll_fps),
            "adaptive_drop": bool(fps_mon.drop_next),
            "events": (["fall"] if is_fall else []) +
                      (["zone_breach"] if is_breach else []) +
                      (["reaching"] if is_reach else []) +
                      (["crowd_alert"] if crowd_alert else []),
            "landmarks": [{"id": j, "x": float(round(x,4)), "y": float(round(y,4))}
                          for j, (x, y) in enumerate(kp_first)] if kp_first else [],
        })
        
        if (i+1) % 30 == 0:
            print(f"[{i+1:>3}/{N}] FPS:{roll_fps:.1f} Falls:{fall_c} "
                  f"Breach:{breach_c} Reach:{reach_c} Drops:{fps_mon.drops}")
    
    writer.release()
    json_path = os.path.join(OUTPUT_DIR, f"alerts_{name}.json")
    with open(json_path, "w") as f:
        json.dump(log, f, indent=2)
    
    elapsed = time.time() - t0
    sensitivity = tp/(tp+fn) if (tp+fn) > 0 else 0
    specificity = tn/(tn+fp) if (tn+fp) > 0 else 0
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0
    f1 = (2*precision*sensitivity/(precision+sensitivity) if (precision+sensitivity) > 0 else 0)
    accuracy = (tp+tn)/N if N > 0 else 0
    avg_fps = float(np.mean(fps_log))
    
    result = {
        "sequence": name,
        "frames": N,
        "elapsed_s": round(elapsed, 1),
        "avg_fps": round(avg_fps, 1),
        "adaptive_drops": fps_mon.drops,
        "fall": {
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "sensitivity": round(sensitivity, 3),
            "specificity": round(specificity, 3),
            "precision": round(precision, 3),
            "f1": round(f1, 3),
            "accuracy": round(accuracy, 3)
        },
        "events": {
            "falls": fall_c,
            "breaches": breach_c,
            "reaches": reach_c,
            "crowd_alerts": crowd_c
        },
        "video": out_path,
        "json": json_path,
    }
    
    print(f"""
RESULTS — {name}
Time:{elapsed:.1f}s  Avg FPS:{avg_fps:.1f} Drops:{fps_mon.drops}
Fall → TP:{tp} FP:{fp} TN:{tn} FN:{fn}
       Sens:{sensitivity:.3f} Spec:{specificity:.3f}
       Prec:{precision:.3f} F1:{f1:.3f} Acc:{accuracy:.3f}
Events → Fall:{fall_c} Breach:{breach_c} Reach:{reach_c} Crowd:{crowd_c}
Video → {out_path}
""")
    return result

# ============================================================
# RUN ALL SEQUENCES
# ============================================================
all_results = []
for seq in SEQUENCES:
    r = process_sequence(seq)
    if r:
        all_results.append(r)

summary_path = os.path.join(OUTPUT_DIR, "ghost_cam_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n{'='*70}")
print("GHOST-CAM — FINAL SUMMARY (Paper 1 + BLANKET + Edge-Cloud)")
print(f"{'='*70}")
print(f"{'Sequence':<28} {'FPS':>5} {'F1':>6} {'Acc':>6} {'Falls':>6} {'Breach':>7} {'Reach':>6}")
print("-"*68)
for r in all_results:
    print(f"{r['sequence']:<28} "
          f"{r['avg_fps']:>5.1f} "
          f"{r['fall']['f1']:>6.3f} "
          f"{r['fall']['accuracy']:>6.3f} "
          f"{r['events']['falls']:>6} "
          f"{r['events']['breaches']:>7} "
          f"{r['events']['reaches']:>6}")
print("="*70)
print(f"\n✅ Summary JSON → {summary_path}")
print(f"✅ Videos saved in {OUTPUT_DIR}/")


✅ Dataset: /kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb
✅ YOLOv8n-Pose on CPU


  fall-01-cam0-rgb  (160 frames | GT split 60% normal)
[ 30/160] FPS:11.0 Falls:0 Breach:30 Reach:0 Drops:28
[ 60/160] FPS:10.5 Falls:0 Breach:60 Reach:0 Drops:58
[ 90/160] FPS:10.2 Falls:0 Breach:85 Reach:0 Drops:88
[120/160] FPS:10.4 Falls:13 Breach:85 Reach:0 Drops:118
[150/160] FPS:9.8 Falls:43 Breach:85 Reach:2 Drops:148

RESULTS — fall-01-cam0-rgb
Time:17.2s  Avg FPS:10.2 Drops:158
Fall → TP:53 FP:0 TN:96 FN:11
       Sens:0.828 Spec:1.000
       Prec:1.000 F1:0.906 Acc:0.931
Events → Fall:53 Breach:85 Reach:2 Crowd:0
Video → /kaggle/working/ghost_fall-01-cam0-rgb.avi


  fall-02-cam0-rgb  (110 frames | GT split 60% normal)
[ 30/110] FPS:10.2 Falls:0 Breach:0 Reach:0 Drops:28
[ 60/110] FPS:10.9 Falls:0 Breach:0 Reach:0 Drops:58
[ 90/110] FPS:10.3 Falls:28 Breach:0 Reach:4 Drops:88

RESULTS — fall-02-cam0-rgb
Time:12.0s  Avg FPS:10.1 Drops:108
Fall → TP